# Data Cleaning

This notebook prepares the collected YouTube comments for exploratory analysis and sentiment analysis.

The raw dataset is kept unchanged. Cleaning is performed on a copy of the raw data, and the resulting dataset is saved to `data/processed/`.

The cleaning process focuses on:
- validating the collected data
- standardizing timestamps
- removing unnecessary whitespace
- handling URLs and HTML entities where appropriate
- preserving emojis and sentiment-bearing punctuation
- checking for duplicate or empty comments
- creating basic text features

In [2]:
import pandas as pd
import re
from html import unescape
from pathlib import Path

In [3]:
input_path = Path("../data/raw/youtube_comments_750.csv")

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (750, 8)


,domain,creator,video_id,video_title,comment_id,comment_text,like_count,published_at
0,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgzKqfHI8Eags5rqHVJ4AaABAg,Learned a lot of lessons from the last time I ...,4919,2026-09-18T15:56:48Z
1,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,Ugw2CZt8wqvMWpkUvml4AaABAg,18 pro series reporting lot of problems 😂pink ...,0,2026-09-26T17:23:15Z
2,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgxUGyHsxYZkzk7qHuF4AaABAg,What game is that??? 3:16,0,2026-09-26T16:45:06Z
3,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgyQxd36wlTmY5jj9_F4AaABAg,It's so cherry picking and most of the points ...,0,2026-09-26T16:19:00Z
4,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,Ugwh_zxyYLUROYEJKwF4AaABAg,"You’ve never had thermal issues?? Holy cow, I’...",0,2026-09-26T15:12:19Z


In [4]:
clean_df = df.copy()

clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   domain        750 non-null    str  
 1   creator       750 non-null    str  
 2   video_id      750 non-null    str  
 3   video_title   750 non-null    str  
 4   comment_id    750 non-null    str  
 5   comment_text  750 non-null    str  
 6   like_count    750 non-null    int64
 7   published_at  750 non-null    str  
dtypes: int64(1), str(7)
memory usage: 205.2 KB


In [5]:
print("Missing values:")
print(clean_df.isnull().sum())

print("\nDuplicate rows:", clean_df.duplicated().sum())
print("Duplicate comment IDs:", clean_df["comment_id"].duplicated().sum())

Missing values:
domain          0
creator         0
video_id        0
video_title     0
comment_id      0
comment_text    0
like_count      0
published_at    0
dtype: int64

Duplicate rows: 0
Duplicate comment IDs: 0


In [6]:
pd.set_option("display.max_colwidth", 200)

clean_df[
    ["domain", "comment_text"]
].sample(20, random_state=42)

,domain,comment_text
506,cats,Kurt is loafing there like a ginger bread😂\nThe cats in the shelter will love it ❤
357,travel,Hello from Flagstaff! My girl and i are huge fans man! Love the videos! So exciting to see u get joyride form out target haha! Keep up the amazing videos!!❤
133,technology,I got one for $13 bucks on whatnot from dailydealzga fire guy to buy from 🔥
250,travel,Hidden message of the video.\nIt ALL ends with JESUS.
299,travel,I got 50/48🎉
680,cats,They dont sell in Canada if anyone can tell me where i could find this please DM me ❤
336,travel,I was in here😊😊😊😊
155,technology,"the 24mp output off nine 200mp shots is a waste, that stitching should've been the whole point. feels like the gimbal got finished before the software did"
528,cats,I thought this was sambucha lmao 😭😭
736,cats,What I've learned about my cats is that you never give them treats straight away. You touch the button and say good boy then give them a pet then a treat. This seems to work for my cats.


## Text Cleaning Strategy

The cleaning process is intentionally conservative because punctuation, emojis, capitalization, and short expressions can contain useful sentiment information.

The original `comment_text` column is preserved. A separate `clean_text` column is created for analysis.

The following transformations are applied:
- Decode HTML entities
- Remove URLs
- Normalize excessive whitespace
- Remove leading and trailing whitespace

Emojis, punctuation, and short comments are retained because they may contribute to sentiment classification.

In [8]:
def clean_comment(text):
    text = unescape(str(text))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
clean_df["clean_text"] = clean_df["comment_text"].apply(clean_comment)

In [10]:
clean_df[
    ["comment_text", "clean_text"]
].sample(20, random_state=42)

,comment_text,clean_text
506,Kurt is loafing there like a ginger bread😂\nThe cats in the shelter will love it ❤,Kurt is loafing there like a ginger bread😂 The cats in the shelter will love it ❤
357,Hello from Flagstaff! My girl and i are huge fans man! Love the videos! So exciting to see u get joyride form out target haha! Keep up the amazing videos!!❤,Hello from Flagstaff! My girl and i are huge fans man! Love the videos! So exciting to see u get joyride form out target haha! Keep up the amazing videos!!❤
133,I got one for $13 bucks on whatnot from dailydealzga fire guy to buy from 🔥,I got one for $13 bucks on whatnot from dailydealzga fire guy to buy from 🔥
250,Hidden message of the video.\nIt ALL ends with JESUS.,Hidden message of the video. It ALL ends with JESUS.
299,I got 50/48🎉,I got 50/48🎉
680,They dont sell in Canada if anyone can tell me where i could find this please DM me ❤,They dont sell in Canada if anyone can tell me where i could find this please DM me ❤
336,I was in here😊😊😊😊,I was in here😊😊😊😊
155,"the 24mp output off nine 200mp shots is a waste, that stitching should've been the whole point. feels like the gimbal got finished before the software did","the 24mp output off nine 200mp shots is a waste, that stitching should've been the whole point. feels like the gimbal got finished before the software did"
528,I thought this was sambucha lmao 😭😭,I thought this was sambucha lmao 😭😭
736,What I've learned about my cats is that you never give them treats straight away. You touch the button and say good boy then give them a pet then a treat. This seems to work for my cats.,What I've learned about my cats is that you never give them treats straight away. You touch the button and say good boy then give them a pet then a treat. This seems to work for my cats.


In [11]:
print(
    clean_df["comment_text"]
    .str.contains(r"\\n", regex=True, na=False)
    .sum()
)

0


In [12]:
clean_df[
    ["comment_text", "clean_text"]
].sample(20, random_state=42)

,comment_text,clean_text
506,Kurt is loafing there like a ginger bread😂\nThe cats in the shelter will love it ❤,Kurt is loafing there like a ginger bread😂 The cats in the shelter will love it ❤
357,Hello from Flagstaff! My girl and i are huge fans man! Love the videos! So exciting to see u get joyride form out target haha! Keep up the amazing videos!!❤,Hello from Flagstaff! My girl and i are huge fans man! Love the videos! So exciting to see u get joyride form out target haha! Keep up the amazing videos!!❤
133,I got one for $13 bucks on whatnot from dailydealzga fire guy to buy from 🔥,I got one for $13 bucks on whatnot from dailydealzga fire guy to buy from 🔥
250,Hidden message of the video.\nIt ALL ends with JESUS.,Hidden message of the video. It ALL ends with JESUS.
299,I got 50/48🎉,I got 50/48🎉
680,They dont sell in Canada if anyone can tell me where i could find this please DM me ❤,They dont sell in Canada if anyone can tell me where i could find this please DM me ❤
336,I was in here😊😊😊😊,I was in here😊😊😊😊
155,"the 24mp output off nine 200mp shots is a waste, that stitching should've been the whole point. feels like the gimbal got finished before the software did","the 24mp output off nine 200mp shots is a waste, that stitching should've been the whole point. feels like the gimbal got finished before the software did"
528,I thought this was sambucha lmao 😭😭,I thought this was sambucha lmao 😭😭
736,What I've learned about my cats is that you never give them treats straight away. You touch the button and say good boy then give them a pet then a treat. This seems to work for my cats.,What I've learned about my cats is that you never give them treats straight away. You touch the button and say good boy then give them a pet then a treat. This seems to work for my cats.


In [13]:
empty_clean = clean_df["clean_text"].eq("").sum()

print("Empty cleaned comments:", empty_clean)

Empty cleaned comments: 0


In [14]:
changed = (clean_df["comment_text"] != clean_df["clean_text"]).sum()

print("Comments changed by cleaning:", changed)

Comments changed by cleaning: 87


In [15]:
clean_df["character_count"] = clean_df["clean_text"].str.len()

clean_df["word_count"] = clean_df["clean_text"].str.split().str.len()

In [16]:
clean_df[
    ["clean_text", "character_count", "word_count"]
].head(10)

,clean_text,character_count,word_count
0,Learned a lot of lessons from the last time I did this dual personality thing. Let me know how you found it!,108,22
1,18 pro series reporting lot of problems 😂pink line issues durability issues,75,12
2,What game is that??? 3:16,25,5
3,It's so cherry picking and most of the points are easy to crack. Battery point is well taken. But imagine how much longer life it will get if it gets the silicon carbon battery with power efficien...,841,150
4,"You’ve never had thermal issues?? Holy cow, I’ve had thermal issues for the past two generations and I hardly do any gaming.",124,22
5,My IPhone 18 pro max keeps rebooting itself and freezing for some seconds,73,13
6,Sponsored by Apple! You did a better job convincing someone to buy this than Apple!,83,15
7,"Samsung Privacy display ""feature"" is such a scam. I got the same privacy display by buying an 10 EURO screen protector sticker. Literally the same effect. But the sticker actually provides actual ...",260,42
8,so i might get cancer in future i should start stocking up medicine and chemo therapy medics now cause who knows what will happen tommorow got it!,146,27
9,Apple and Oranges….. Yawn …. Make it more expensive so almost intelligent niche people buy it thinking that a phone upgrade makes them stand above others ….. what a sad bubble 😓😓…. Good marketing ...,205,36


In [17]:
clean_df[["character_count", "word_count"]].describe()

,character_count,word_count
count,750.000000,750.000000
mean,97.081333,18.568000
std,156.537851,29.216517
min,1.000000,1.000000
25%,27.250000,5.000000
50%,51.000000,10.000000
75%,114.000000,22.000000
max,2492.000000,466.000000


In [18]:
print("Rows:", len(clean_df))
print("Unique comment IDs:", clean_df["comment_id"].nunique())
print("Duplicate comment IDs:", clean_df["comment_id"].duplicated().sum())

Rows: 750
Unique comment IDs: 750
Duplicate comment IDs: 0


In [19]:
output_path = Path("../data/processed/youtube_comments_clean.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")

Saved cleaned dataset to: ../data/processed/youtube_comments_clean.csv


In [20]:
processed_df = pd.read_csv(output_path)

print("Processed dataset shape:", processed_df.shape)
processed_df.head()

Processed dataset shape: (750, 11)


,domain,creator,video_id,video_title,comment_id,comment_text,like_count,published_at,clean_text,character_count,word_count
0,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgzKqfHI8Eags5rqHVJ4AaABAg,Learned a lot of lessons from the last time I did this dual personality thing. Let me know how you found it!,4919,2026-09-18T15:56:48Z,Learned a lot of lessons from the last time I did this dual personality thing. Let me know how you found it!,108,22
1,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,Ugw2CZt8wqvMWpkUvml4AaABAg,18 pro series reporting lot of problems 😂pink line issues durability issues,0,2026-09-26T17:23:15Z,18 pro series reporting lot of problems 😂pink line issues durability issues,75,12
2,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgxUGyHsxYZkzk7qHuF4AaABAg,What game is that??? 3:16,0,2026-09-26T16:45:06Z,What game is that??? 3:16,25,5
3,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,UgyQxd36wlTmY5jj9_F4AaABAg,It's so cherry picking and most of the points are easy to crack. Battery point is well taken. But imagine how much longer life it will get if it gets the silicon carbon battery with power efficien...,0,2026-09-26T16:19:00Z,It's so cherry picking and most of the points are easy to crack. Battery point is well taken. But imagine how much longer life it will get if it gets the silicon carbon battery with power efficien...,841,150
4,technology,Mrwhosetheboss,PjAyUhf52D0,iPhone 18 Pro Review - My Last iPhone?,Ugwh_zxyYLUROYEJKwF4AaABAg,"You’ve never had thermal issues?? Holy cow, I’ve had thermal issues for the past two generations and I hardly do any gaming.",0,2026-09-26T15:12:19Z,"You’ve never had thermal issues?? Holy cow, I’ve had thermal issues for the past two generations and I hardly do any gaming.",124,22


## Cleaning Summary

The raw YouTube comment dataset contained 750 comments across 15 videos and three domains.

The cleaning process:
- preserved the original `comment_text`
- created a separate `clean_text` column
- decoded HTML entities
- removed URLs
- normalized whitespace
- preserved emojis and punctuation
- retained short comments
- verified that no comments became empty
- verified that all comment IDs remained unique
- added character and word count features

No rows were removed during the cleaning process.
The cleaned dataset was saved to `data/processed/youtube_comments_clean.csv`.